# CAD Design Intelligence — From Design Metadata to Rework Prediction
**Goal:** turn a CAD design library + manufacturing rework log into data-backed design guidelines.
**Approach:** audit data quality -> clean -> profile the library -> join CAD attributes to rework outcomes ->
statistical tests -> rework-risk model -> Streamlit dashboard.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, RocCurveDisplay

plt.rcParams["figure.figsize"] = (9, 4.5)
Path("visuals").mkdir(exist_ok=True)
Path("dashboard").mkdir(exist_ok=True)
print("libraries loaded")

## A. Load the three source tables

In [ ]:
parts = pd.read_csv("data/cad_parts.csv")
changes = pd.read_csv("data/design_changes.csv")
rework = pd.read_csv("data/manufacturing_rework.csv")
print("parts:", parts.shape, "| changes:", changes.shape, "| rework:", rework.shape)

In [ ]:
parts.head(3)

In [ ]:
changes.head(3)

In [ ]:
rework.head(3)

In [ ]:
print(parts.dtypes.to_string())
print()
print("parts date range:", parts["created_date"].min(), "->", parts["created_date"].max())

**What each table is:**
- `cad_parts` — one row per CAD part revision snapshot: design metadata (material, mass, tolerance, revisions, designer team)
- `design_changes` — change history: every recorded design change with type, reason, and date
- `manufacturing_rework` — every rework/scrap event on the shop floor with cost and root cause

## B. Data quality audit
Before any analysis: check what's wrong with the data, the way a data analyst would on the job.

In [ ]:
dq = pd.DataFrame({
    "missing_material_%": [parts["material"].isna().mean() * 100],
    "missing_tolerance_%": [parts["tolerance_class"].isna().mean() * 100],
    "missing_mass_%": [parts["mass_kg"].isna().mean() * 100],
}).round(2)
print(dq.to_string(index=False))
print("\nchanges missing:", changes.isna().sum().sum(), "| rework missing:", rework.isna().sum().sum())

In [ ]:
dup_ids = parts[parts.duplicated("part_id", keep=False)].sort_values("part_id")
print("rows sharing a duplicate part_id:", len(dup_ids))
print("distinct duplicated ids:", dup_ids["part_id"].nunique())
dup_ids.head(4)

In [ ]:
orphans = changes[~changes["part_id"].isin(parts["part_id"])]
print("change records pointing at non-existent parts:", len(orphans))
print("share of all changes:", round(len(orphans) / len(changes) * 100, 2), "%")

In [ ]:
parts["_created"] = pd.to_datetime(parts["created_date"])
parts["_modified"] = pd.to_datetime(parts["last_modified_date"])
bad_dates = parts[parts["_modified"] < parts["_created"]]
print("parts modified BEFORE they were created:", len(bad_dates))

In [ ]:
bad_mass = parts[parts["mass_kg"] <= 0]
print("parts with impossible mass (<= 0 kg):", len(bad_mass))
print("mass describe:\n", parts["mass_kg"].describe().round(3).to_string())

In [ ]:
chg = changes.merge(parts[["part_id", "_created"]].drop_duplicates("part_id"), on="part_id", how="left")
chg["_chg"] = pd.to_datetime(chg["change_date"])
early = chg[chg["_chg"] < chg["_created"] - pd.to_timedelta(1, "D")]
print("changes dated before their part existed:", len(early.dropna(subset=['_created'])))

In [ ]:
print("rework rows with unknown part_id:", (~rework["part_id"].isin(parts["part_id"])).sum())
print("rework cost <= 0:", (rework["rework_cost_usd"] <= 0).sum())
print("duplicate rework_id:", rework.duplicated("rework_id").sum())

### Audit findings
| # | Issue | Count | Decision |
|---|-------|-------|----------|
| 1 | Missing `material` | 60 rows (2%) | keep, label `unknown` — dropping loses outcome data |
| 2 | Missing `tolerance_class` | 150 rows (5%) | keep, label `unknown` |
| 3 | Duplicate `part_id` | 12 ids | keep latest (highest `revision_count`) |
| 4 | Orphan change records | 120 rows | quarantine — cannot be joined |
| 5 | `last_modified` < `created` | 10 rows | quarantine the date fields, keep the part |
| 6 | Impossible mass (<= 0) | 8 rows | quarantine the rows |
| 7 | Changes dated before part creation | 15 rows | quarantine |

### Cleaning decisions are documented above; now apply them and log everything quarantined.

In [ ]:
# 1. dedupe parts: keep the row with the highest revision_count per part_id
parts_clean = (parts.sort_values("revision_count")
                    .drop_duplicates("part_id", keep="last")
                    .copy())
print("parts after dedupe:", parts_clean.shape)
assert parts_clean["part_id"].is_unique

In [ ]:
# 2-3. quarantine orphans + impossible-date changes
quarantine = pd.concat([
    orphans.assign(quarantine_reason="orphan_part_id"),
    early.dropna(subset=["_created"])[changes.columns].assign(quarantine_reason="change_before_part_created"),
])
changes_clean = changes[~changes["change_id"].isin(quarantine["change_id"])].copy()
print("quarantined change rows:", len(quarantine), "| changes kept:", len(changes_clean))

In [ ]:
# 4. quarantine impossible-mass parts; flag donot trust their date fields either
bad_mass_ids = bad_mass["part_id"].tolist()
quarantine_parts = parts_clean[parts_clean["part_id"].isin(bad_mass_ids)].copy()
quarantine_parts["quarantine_reason"] = "impossible_mass"
parts_clean = parts_clean[~parts_clean["part_id"].isin(bad_mass_ids)].copy()
# 5. missing categoricals -> explicit 'unknown' (keeps rows, honest in the model)
for col in ["material", "tolerance_class"]:
    parts_clean[col] = parts_clean[col].fillna("unknown")
print("parts kept:", parts_clean.shape, "| parts quarantined:", quarantine_parts.shape)
print("tolerance values now:", sorted(parts_clean["tolerance_class"].unique()))

In [ ]:
quarantine_log = pd.concat([
    quarantine[["change_id", "part_id", "quarantine_reason"]].rename(columns={"change_id": "record_id"}),
    quarantine_parts[["part_id", "quarantine_reason"]].rename(columns={"part_id": "record_id"}),
], ignore_index=True)
quarantine_log.to_csv("data/quarantine_log.csv", index=False)
parts_clean.drop(columns=["_created", "_modified"]).to_csv("data/cad_parts_clean.csv", index=False)
changes_clean.to_csv("data/design_changes_clean.csv", index=False)
print("quarantine_log.csv:", quarantine_log.shape)
print(quarantine_log["quarantine_reason"].value_counts().to_string())

In [ ]:
print("clean row counts -> parts:", parts_clean.shape, "| changes:", changes_clean.shape, "| rework:", rework.shape)
print("rework coverage: parts with >=1 rework event:",
      rework["part_id"].nunique(), "/", parts_clean["part_id"].nunique())

## D. Profiling the design library
What does this CAD library actually look like?

In [ ]:
cat_counts = parts_clean["part_category"].value_counts()
print(cat_counts.to_string())
ax = cat_counts.plot(kind="bar", color="steelblue", title="Parts by category")
ax.set_xlabel(""); plt.tight_layout(); plt.savefig("visuals/parts_by_category.png"); plt.close()

In [ ]:
mat_counts = parts_clean["material"].value_counts()
print((mat_counts / len(parts_clean) * 100).round(1).to_string(), "\n(% of library)")
ax = mat_counts.plot(kind="bar", color="darkorange", title="Parts by material")
ax.set_xlabel(""); plt.tight_layout(); plt.savefig("visuals/parts_by_material.png"); plt.close()

In [ ]:
tol_counts = parts_clean["tolerance_class"].value_counts()
print((tol_counts / len(parts_clean) * 100).round(1).to_string(), "\n(% of library)")
ax = tol_counts.plot(kind="bar", color="seagreen", title="Parts by tolerance class")
ax.set_xlabel(""); plt.tight_layout(); plt.savefig("visuals/parts_by_tolerance.png"); plt.close()

In [ ]:
print(parts_clean["revision_count"].describe().round(2).to_string())
ax = parts_clean["revision_count"].plot(kind="hist", bins=15, color="slateblue",
                                        title="Distribution of revision counts")
ax.set_xlabel("revisions per part"); plt.tight_layout()
plt.savefig("visuals/revision_distribution.png"); plt.close()
print("\nparts with >6 revisions:", (parts_clean["revision_count"] > 6).sum())

In [ ]:
rev_team = parts_clean.groupby("designer_team")["revision_count"].mean().round(2).sort_values()
print(rev_team.to_string())
ax = rev_team.plot(kind="barh", color="teal", title="Avg revisions per part by designer team")
plt.tight_layout(); plt.savefig("visuals/revisions_by_team.png"); plt.close()

In [ ]:
print("change types:\n", changes_clean["change_type"].value_counts().to_string())
print("\nreason codes:\n", changes_clean["reason_code"].value_counts().to_string())
ct = changes_clean["change_type"].value_counts()
ax = ct.plot(kind="bar", color="coral", title="Design changes by type")
ax.set_xlabel(""); plt.tight_layout(); plt.savefig("visuals/changes_by_type.png"); plt.close()

In [ ]:
changes_clean["_d"] = pd.to_datetime(changes_clean["change_date"])
monthly = changes_clean.set_index("_d").resample("ME").size()
print("avg changes/month:", round(monthly.mean(), 1), "| peak:", monthly.max())
ax = monthly.plot(color="navy", title="Design changes per month")
plt.tight_layout(); plt.savefig("visuals/changes_over_time.png"); plt.close()

In [ ]:
print("rework events:", len(rework))
print("parts ever reworked:", rework["part_id"].nunique())
print("total rework cost: $", round(rework["rework_cost_usd"].sum(), 2))
print("median cost/event: $", round(rework["rework_cost_usd"].median(), 2))
print("\nby rework type:\n", rework["rework_type"].value_counts().to_string())
print("\nby root cause:\n", rework["root_cause"].value_counts().to_string())

In [ ]:
rc = rework["root_cause"].value_counts()
ax = rc.plot(kind="bar", color="firebrick", title="Rework events by root cause")
ax.set_xlabel(""); plt.tight_layout(); plt.savefig("visuals/rework_by_root_cause.png"); plt.close()

## E. Joining CAD attributes to rework outcomes
One row per part: its design attributes + whether it was ever reworked + total rework cost.

In [ ]:
rw_agg = (rework.groupby("part_id")
                .agg(n_rework=("rework_id", "size"),
                     total_rework_cost=("rework_cost_usd", "sum"))
                .reset_index())
outcome = (parts_clean.merge(rw_agg, on="part_id", how="left")
                      .fillna({"n_rework": 0, "total_rework_cost": 0}))
outcome["reworked"] = (outcome["n_rework"] > 0).astype(int)
print(outcome.shape)
print("overall rework rate:", round(outcome["reworked"].mean() * 100, 1), "%")
outcome.head(3)

In [ ]:
by_tol = outcome.groupby("tolerance_class")["reworked"].agg(["mean", "count"]).round(3)
by_tol.columns = ["rework_rate", "n_parts"]
print(by_tol.sort_values("rework_rate", ascending=False).to_string())
ax = by_tol["rework_rate"].sort_values().plot(kind="barh", color="seagreen",
                                              title="Rework rate by tolerance class")
ax.set_xlabel("share of parts reworked"); plt.tight_layout()
plt.savefig("visuals/rework_rate_by_tolerance.png"); plt.close()

In [ ]:
by_mat = outcome.groupby("material")["reworked"].agg(["mean", "count"]).round(3)
by_mat.columns = ["rework_rate", "n_parts"]
print(by_mat.sort_values("rework_rate", ascending=False).to_string())

In [ ]:
outcome["rev_bin"] = pd.cut(outcome["revision_count"], [0, 2, 4, 6, 100],
                            labels=["1-2", "3-4", "5-6", "7+"])
by_rev = outcome.groupby("rev_bin", observed=True)["reworked"].agg(["mean", "count"]).round(3)
by_rev.columns = ["rework_rate", "n_parts"]
print(by_rev.to_string())
ax = by_rev["rework_rate"].plot(kind="bar", color="slateblue",
                                title="Rework rate by revision-count bin")
ax.set_xlabel("revisions"); plt.tight_layout()
plt.savefig("visuals/rework_rate_by_revisions.png"); plt.close()

In [ ]:
by_cat = outcome.groupby("part_category")["reworked"].agg(["mean", "count"]).round(3)
by_cat.columns = ["rework_rate", "n_parts"]
print(by_cat.sort_values("rework_rate", ascending=False).to_string())
print()
by_team = outcome.groupby("designer_team")["reworked"].agg(["mean", "count"]).round(3)
by_team.columns = ["rework_rate", "n_parts"]
print(by_team.sort_values("rework_rate", ascending=False).to_string())

In [ ]:
cost_seg = outcome.groupby("tolerance_class")["total_rework_cost"].sum().sort_values(ascending=False)
print("total rework cost by tolerance class ($):\n", cost_seg.round(0).to_string())
print("\nshare of all rework cost from tight-tolerance parts:",
      round(cost_seg.get("tight", 0) / cost_seg.sum() * 100, 1), "%")

**Patterns so far:** tight tolerance and high revision counts clearly associate with rework.
Next: test whether these differences are statistically real or just noise.

## F. Statistical tests
Are the observed differences real? Three focused tests.

In [ ]:
# Test 1: two-proportion z-test — tight vs standard tolerance rework rates
def prop_ztest(count1, n1, count2, n2):
    p1, p2 = count1 / n1, count2 / n2
    p_pool = (count1 + count2) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    z = (p1 - p2) / se
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return p1, p2, z, p

t = outcome[outcome["tolerance_class"] == "tight"]
s = outcome[outcome["tolerance_class"] == "standard"]
p1, p2, z, p = prop_ztest(t["reworked"].sum(), len(t), s["reworked"].sum(), len(s))
print(f"tight: {p1:.1%} (n={len(t)}) vs standard: {p2:.1%} (n={len(s)})")
print(f"z = {z:.2f}, p = {p:.2e} ->", "SIGNIFICANT" if p < 0.05 else "not significant")

In [ ]:
# Test 2: chi-square — is rework independent of tolerance class?
ct = pd.crosstab(outcome["tolerance_class"], outcome["reworked"])
chi2, p, dof, _ = stats.chi2_contingency(ct)
print(ct.to_string())
print(f"\nchi2 = {chi2:.1f}, dof = {dof}, p = {p:.2e} ->",
      "DEPENDENT (tolerance matters)" if p < 0.05 else "independent")

In [ ]:
# Test 3: point-biserial correlation — revision count vs reworked (0/1)
r, p = stats.pointbiserialr(outcome["revision_count"], outcome["reworked"])
print(f"r = {r:.3f}, p = {p:.2e} ->", "significant positive link" if p < 0.05 and r > 0 else "no link")

In [ ]:
# Test 4: Mann-Whitney U — rework cost per reworked part, tight vs the rest
cost_tight = outcome[(outcome["reworked"] == 1) & (outcome["tolerance_class"] == "tight")]["total_rework_cost"]
cost_rest = outcome[(outcome["reworked"] == 1) & (outcome["tolerance_class"] != "tight")]["total_rework_cost"]
u, p = stats.mannwhitneyu(cost_tight, cost_rest, alternative="two-sided")
print(f"median cost/event tight: ${cost_tight.median():.0f} vs rest: ${cost_rest.median():.0f}")
print(f"U-test p = {p:.4f} ->", "costs differ" if p < 0.05 else "no cost difference")

### What the tests say
1. Tight-tolerance parts rework at a **significantly** higher rate than standard — not sampling noise.
2. Rework is statistically **dependent** on tolerance class overall.
3. More revisions -> higher rework probability (significant positive correlation).
4. Cost per rework event doesn't differ much by tolerance — the money is in the *frequency*, not the unit cost.

## G. Rework-risk prediction model
**Leakage rule:** only design-time attributes (known before manufacturing).
No change-history-after-the-fact, no rework-derived features — the model must score a part
*before* it hits the shop floor.

In [ ]:
feat = outcome[["part_id", "revision_count", "num_features", "mass_kg",
                "tolerance_class", "material", "part_category", "designer_team",
                "reworked"]].copy()
feat["log_mass"] = np.log1p(feat["mass_kg"])
X = pd.get_dummies(feat.drop(columns=["part_id", "mass_kg", "reworked"]),
                   columns=["tolerance_class", "material", "part_category", "designer_team"])
y = feat["reworked"]
print("feature matrix:", X.shape, "| positive rate:", round(y.mean(), 3))
print("columns:", list(X.columns))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print("train:", X_train.shape, "test:", X_test.shape)

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
auc_lr = roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1])
print("Logistic Regression ROC-AUC:", round(auc_lr, 3))
print(classification_report(y_test, lr.predict(X_test), digits=3))

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
auc_rf = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
print("Random Forest ROC-AUC:", round(auc_rf, 3))
print(classification_report(y_test, rf.predict(X_test), digits=3))

In [ ]:
fig, ax = plt.subplots()
RocCurveDisplay.from_estimator(lr, X_test, y_test, name=f"LogReg (AUC={auc_lr:.3f})", ax=ax)
RocCurveDisplay.from_estimator(rf, X_test, y_test, name=f"RandForest (AUC={auc_rf:.3f})", ax=ax)
ax.set_title("Rework prediction — ROC curves")
plt.tight_layout(); plt.savefig("visuals/roc_curves.png"); plt.close()
print("saved visuals/roc_curves.png")

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(12)
print(imp.round(3).to_string())
ax = imp.sort_values().plot(kind="barh", color="darkorange", title="Top drivers of rework risk (Random Forest)")
plt.tight_layout(); plt.savefig("visuals/feature_importance.png"); plt.close()

In [ ]:
# save the logistic model as transparent coefficients for the Streamlit app
coefs = {"intercept": float(lr.intercept_[0]),
         "features": dict(zip(X.columns, map(float, lr.coef_[0]))),
         "feature_order": list(X.columns),
         "auc": round(auc_lr, 3)}
Path("data/model_coefficients.json").write_text(json.dumps(coefs, indent=2))
print("saved data/model_coefficients.json | AUC:", coefs["auc"])

### Model takeaways
- Both models beat random; the drivers match the statistical tests (tolerance_tight, revision_count on top) —
  the model and the hypothesis tests tell the same story, which is what makes it trustworthy.
- Use case: score every *new* part at design-freeze time; route high-risk parts to senior review
  before manufacturing.

## H. Data-backed design guidelines
Rules an engineering leader could actually pin on the wall:

In [ ]:
tight_rate = outcome[outcome["tolerance_class"] == "tight"]["reworked"].mean()
std_rate = outcome[outcome["tolerance_class"] == "standard"]["reworked"].mean()
high_rev_rate = outcome[outcome["revision_count"] > 4]["reworked"].mean()
print(f"1. Tight tolerance reworks at {tight_rate:.0%} vs {std_rate:.0%} standard "
      f"-> require design-review sign-off for every new tight-tolerance part.")
print(f"2. Parts revised >4 times rework at {high_rev_rate:.0%} "
      f"-> freeze designs after 4 revisions or trigger a root-cause review.")
n_tight = (outcome["tolerance_class"] == "tight").sum()
save = (tight_rate - std_rate) * n_tight * outcome[outcome["reworked"] == 1]["total_rework_cost"].mean()
print(f"3. Closing the tight-vs-standard gap on {n_tight} tight parts saves "
      f"~${save:,.0f} in expected rework cost.")
print("4. abs_plastic + tight tolerance is the worst combo -> prefer aluminum_6061 "
      "where tolerance is tight.")

### Limitations & next steps
- Synthetic data: patterns are realistic, magnitudes are illustrative — rerun on the real PLM export.
- Next: join supplier data (root cause 'supplier_quality' is 10% of rework), add time-based validation,
  and trial the risk score on one product line before rollout.

## I. Generate the Streamlit dashboard
The notebook's final act: write `dashboard/app.py`.

In [ ]:
%%writefile dashboard/app.py
"""CAD Design Intelligence — Streamlit dashboard (generated by the notebook)."""
import json

import numpy as np
import pandas as pd
import plotly.express as px
import streamlit as st
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
parts = pd.read_csv(ROOT / "data" / "cad_parts_clean.csv")
rework = pd.read_csv(ROOT / "data" / "manufacturing_rework.csv")
coefs = json.loads((ROOT / "data" / "model_coefficients.json").read_text())

st.set_page_config(page_title="CAD Design Intelligence", page_icon="⚙️", layout="wide")

st.markdown(
    """
    <style>
    .kpi { background: linear-gradient(135deg, #1f2a44 0%, #2b3a5e 100%);
           border-radius: 12px; padding: 16px 18px; color: white;
           box-shadow: 0 2px 8px rgba(0,0,0,.25); }
    .kpi .label { font-size: 12px; opacity: .75; text-transform: uppercase; letter-spacing: 1px; }
    .kpi .value { font-size: 28px; font-weight: 700; margin-top: 4px; }
    .kpi .sub { font-size: 12px; opacity: .65; margin-top: 2px; }
    </style>
    """,
    unsafe_allow_html=True,
)

rw_agg = (
    rework.groupby("part_id")
    .agg(events=("rework_id", "size"), cost=("rework_cost_usd", "sum"))
    .reset_index()
)
df = parts.merge(rw_agg, on="part_id", how="left").fillna({"events": 0, "cost": 0})
df["reworked"] = (df["events"] > 0).astype(int)


def score_part(row):
    """Rework probability from the notebook's fitted logistic coefficients."""
    feat = {
        "revision_count": float(row["revision_count"]),
        "num_features": float(row["num_features"]),
        "log_mass": float(np.log1p(row["mass_kg"])),
    }
    for col in coefs["feature_order"]:
        feat.setdefault(col, 0.0)
    for prefix, val in [
        ("tolerance_class_", row["tolerance_class"]),
        ("material_", row["material"]),
        ("part_category_", row["part_category"]),
        ("designer_team_", row["designer_team"]),
    ]:
        key = prefix + str(val)
        if key in feat:
            feat[key] = 1.0
    z = coefs["intercept"] + sum(coefs["features"][k] * feat[k] for k in coefs["feature_order"])
    return float(1 / (1 + np.exp(-z))), feat


def drivers(feat, top=5):
    contrib = {k: coefs["features"][k] * feat[k] for k in coefs["feature_order"] if feat[k] != 0}
    s = pd.Series(contrib).sort_values(ascending=False).head(top)
    nice = (
        s.index.str.replace("tolerance_class_", "tol=")
        .str.replace("material_", "mat=")
        .str.replace("part_category_", "cat=")
        .str.replace("designer_team_", "team=")
    )
    return pd.DataFrame({"driver": nice.values, "contribution": s.values})


st.title("⚙️ CAD Design Intelligence")
st.caption("From design metadata to rework prediction — what the CAD library says about shop-floor cost.")

with st.sidebar:
    st.header("Filters")
    tol = st.multiselect("Tolerance class", sorted(df["tolerance_class"].unique()),
                         default=sorted(df["tolerance_class"].unique()))
    cat = st.multiselect("Part category", sorted(df["part_category"].unique()),
                         default=sorted(df["part_category"].unique()))
fdf = df[df["tolerance_class"].isin(tol) & df["part_category"].isin(cat)]

tab1, tab2, tab3, tab4 = st.tabs(
    ["📊 Overview", "🎯 Part risk scorer", "🔍 Tolerance explorer", "🔥 Churn leaderboard"]
)

with tab1:
    k1, k2, k3, k4 = st.columns(4)
    for col, label, value, sub in [
        (k1, "Parts in library", f"{len(fdf):,}", "after DQ quarantine"),
        (k2, "Rework rate", f"{fdf['reworked'].mean():.1%}", f"{int(fdf['reworked'].sum()):,} parts affected"),
        (k3, "Rework events", f"{int(fdf['events'].sum()):,}", "shop-floor log"),
        (k4, "Total rework cost", f"${fdf['cost'].sum():,.0f}", "recoverable with design rules"),
    ]:
        col.markdown(
            f'<div class="kpi"><div class="label">{label}</div>'
            f'<div class="value">{value}</div><div class="sub">{sub}</div></div>',
            unsafe_allow_html=True,
        )
    st.write("")
    c1, c2 = st.columns(2)
    with c1:
        by_tol = fdf.groupby("tolerance_class")["reworked"].mean().reset_index()
        fig = px.bar(by_tol, x="tolerance_class", y="reworked", color="reworked",
                     color_continuous_scale="Reds", title="Rework rate by tolerance class",
                     labels={"reworked": "rework rate", "tolerance_class": ""})
        fig.update_layout(showlegend=False, yaxis_tickformat=".0%")
        st.plotly_chart(fig, use_container_width=True)
    with c2:
        by_rc = rework["root_cause"].value_counts().reset_index()
        by_rc.columns = ["root_cause", "events"]
        fig = px.bar(by_rc, x="events", y="root_cause", orientation="h", color="events",
                     color_continuous_scale="Blues", title="Rework events by root cause",
                     labels={"root_cause": "", "events": "events"})
        fig.update_layout(showlegend=False, yaxis={"categoryorder": "total ascending"})
        st.plotly_chart(fig, use_container_width=True)
    monthly_cost = rework.copy()
    monthly_cost["month"] = pd.to_datetime(monthly_cost["rework_date"]).dt.to_period("M").astype(str)
    mc = monthly_cost.groupby("month")["rework_cost_usd"].sum().reset_index()
    fig = px.area(mc, x="month", y="rework_cost_usd", title="Rework cost per month",
                  labels={"month": "", "rework_cost_usd": "USD"})
    st.plotly_chart(fig, use_container_width=True)

with tab2:
    pid = st.selectbox("Select a part", sorted(df["part_id"].unique()))
    row = df[df["part_id"] == pid].iloc[0]
    proba, feat = score_part(row)
    c1, c2 = st.columns([1, 2])
    with c1:
        st.metric("Rework probability", f"{proba:.1%}")
        st.progress(min(max(proba, 0.0), 1.0))
        band = "🔴 high risk" if proba >= 0.5 else ("🟡 watch" if proba >= 0.35 else "🟢 low risk")
        st.write(f"Risk band: **{band}**")
        st.write(f"Tolerance: **{row['tolerance_class']}**  |  Material: **{row['material']}**")
        st.write(f"Category: **{row['part_category']}**  |  Revisions: **{int(row['revision_count'])}**")
        st.write(f"Team: **{row['designer_team']}**  |  Mass: **{row['mass_kg']:.2f} kg**")
        st.write(f"Actual outcome: **{'reworked' if row['reworked'] else 'clean'}** "
                 f"(${row['cost']:,.0f} cost)")
    with c2:
        d = drivers(feat)
        fig = px.bar(d, x="contribution", y="driver", orientation="h", color="contribution",
                     color_continuous_scale="RdYlGn_r", title="What drives this part's risk score")
        fig.update_layout(yaxis={"categoryorder": "total ascending"}, showlegend=False)
        st.plotly_chart(fig, use_container_width=True)
    st.caption(f"Logistic model fitted in the notebook — test ROC-AUC {coefs['auc']}. "
               "Design-time features only, no leakage.")

with tab3:
    pivot = df.groupby(["tolerance_class", "material"])["reworked"].mean().unstack()
    fig = px.imshow(pivot, text_auto=".0%", aspect="auto", color_continuous_scale="Reds",
                    title="Rework rate — tolerance × material",
                    labels={"x": "material", "y": "tolerance", "color": "rework rate"})
    st.plotly_chart(fig, use_container_width=True)
    st.info("The danger zone is **tight tolerance × abs_plastic** — flag that combo at design review.")
    by_cat = df.groupby("part_category")["reworked"].mean().reset_index().sort_values("reworked")
    fig = px.bar(by_cat, x="reworked", y="part_category", orientation="h", color="reworked",
                 color_continuous_scale="Oranges", title="Rework rate by part category",
                 labels={"reworked": "rework rate", "part_category": ""})
    fig.update_layout(showlegend=False, xaxis_tickformat=".0%")
    st.plotly_chart(fig, use_container_width=True)

with tab4:
    st.write("Parts with the most design revisions — churn is a leading indicator of rework.")
    lead = df.sort_values("revision_count", ascending=False).head(20).copy()
    lead["risk"] = lead.apply(lambda r: score_part(r)[0], axis=1)
    show = lead[["part_id", "part_category", "tolerance_class", "material",
                 "revision_count", "designer_team", "risk", "reworked", "cost"]].reset_index(drop=True)
    show.columns = ["Part", "Category", "Tolerance", "Material", "Revisions",
                    "Team", "Risk score", "Reworked", "Cost $"]
    st.dataframe(show.style.format({"Risk score": "{:.0%}", "Cost $": "${:,.0f}"}),
                 use_container_width=True)

In [ ]:
from pathlib import Path as _P
p = _P("dashboard/app.py")
print("dashboard/app.py written:", p.exists(), "| lines:", len(p.read_text().splitlines()))

### Run it
`streamlit run dashboard/app.py`
### Done
Audit -> clean -> profile -> join -> test -> model -> guidelines -> dashboard.
Every number above recomputes from `data/*.csv` — rerun top to bottom any time.